In [ ]:
import pandas as pd
import numpy as np
import tushare as ts
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#先做一些基础设置
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

#然后是交易成本
COMMISSION_RATE = 0.0003        #佣金万三（买入和卖出都收取）
STAMP_TAX = 0.001               #印花税千一（仅卖出时收取）
SLIPPAGE = 0.001                #滑点千一（模拟无法以理想价格成交）

#下面是个人设置，因人而异，可修改部分均列在此
TS_TOKEN = 'your_token'
TS_CODE = '601398.SH'
START_DATE = '20230101'
END_DATE = '20231231'

In [ ]:
#用tushare获取每日数据（当然，自己想要什么数据就用什么函数）
ts.set_token(TS_TOKEN)
pro = ts.pro_api()
#要获取前复权数据，避免因为除权除息导致的价格跳变影响回测结果  
data = pro.daily(ts_code=TS_CODE, start_date=START_DATE, end_date=END_DATE,adjust='qfq')
data_sorted = data.sort_values('trade_date').reset_index(drop=True)

if data_sorted.empty:
    print("No data found for the given parameters.")
else:
    print(data_sorted.head(10))

In [ ]:
def get_stock_board(code):
    try:
        stock_info  = pro.stock_basic(ts_code=code, fields='ts_code,market')
        if not stock_info.empty:
            return stock_info['market'].iloc[0]
        else:
            return "主板"   #如果没查到，保守起见认为是主板
    except Exception as e:
        print(f"Error fetching stock board for {code}: {e}")
        return "主板"       #如果出错，保守起见认为是主板
    
board = get_stock_board(TS_CODE)
print(f"Stock board for {TS_CODE}: {board}")

In [ ]:
#然后我们先搭建一下最简单的双均线策略，看看最基础的量化交易时间序列数据处理方法
data_sorted['return'] = data_sorted['close'].pct_change()
data_sorted['ma5'] = data_sorted['close'].rolling(window=5).mean()
data_sorted['ma10'] = data_sorted['close'].rolling(window=10).mean()

data_sorted['signal'] = 0
data_sorted.loc[data_sorted['ma5'] > data_sorted['ma10'], 'signal'] = 1
data_sorted.loc[data_sorted['ma5'] < data_sorted['ma10'], 'signal'] = -1

print(data_sorted[['close','ma5','ma10','signal']].head(20))

#注意这里的position是指买入卖出信号，1表示买入，-1表示卖出，0表示不操作，shift(1)是为了让信号在第二天才生效，避免未来函数问题
data_sorted['position_shifted'] = data_sorted['signal'].shift(1)

In [ ]:
#不要忘了涨停和跌停无法成交，所以我们还要获取涨停和跌停的价格，避免在这些价格下进行交易
def get_limit_prices(row,df_prev,board):
    pre_close = row['pre_close']
    #如果前收盘价（pre_close）无效（比如上市首日），我们就直接返回None，表示无法计算涨停和跌停价格
    if pd.isna(pre_close) or pre_close <= 0:
        return None, None
    
    try:
        limit_data = pro.stk_limit(ts_code=row['ts_code'], trade_date=row['trade_date'])
        if limit_data is not None and not limit_data.empty:
            limit_up = limit_data['limit_up'].iloc[0]
            limit_down = limit_data['limit_down'].iloc[0]
            return limit_up, limit_down
    except:
        pass    #如果获取涨跌停价格失败，我们就用默认的计算方法
    
    #默认计算
    if board in ['创业板', '科创板']:
        pct = 0.20
    elif board == '北交所':
        pct = 0.30
    else:       #主板和中小板
        pct = 0.10

    up_price = round(pre_close * (1 + pct), 2)
    down_price = round(pre_close * (1 - pct), 2)
    return up_price, down_price

In [ ]:
#然后开始逐日模拟交易
data_sorted['actual_position'] = 0.00
current_position = 0.00

for i, row in data_sorted.iterrows():
    #先按昨天的计划确定今天的仓位
    target = row['position_shifted']
    #如果没有交易信号，就保持原仓位
    if pd.isna(target):
        data_sorted.loc[i, 'actual_position'] = current_position
        continue
    
    #获取涨停和跌停价格
    limit_up, limit_down = get_limit_prices(row, data_sorted.iloc[i-1] if i > 0 else None, board)
    #如果涨停和跌停价格无法获取，就保持原仓位
    if limit_up is None or limit_down is None:
        data_sorted.loc[i, 'actual_position'] = current_position
        continue
    
    #再判断今天是否封死涨停或跌停，我们用严格相等来判断
    is_limit_up = (row['high'] == limit_up) and (row['close'] == limit_up)
    is_limit_down = (row['low'] == limit_down) and (row['close'] == limit_down)
    
    #如果今天封死涨停或跌停，就无法交易
    if is_limit_up or is_limit_down:
        data_sorted.loc[i, 'actual_position'] = current_position
        continue
    
    #如果今天的开盘价为0，说明停牌，无法交易
    if row['open'] == 0:
        data_sorted.loc[i, 'actual_position'] = current_position
        continue
        
    #除去上面的情况，说明今天可以按计划交易，我们就直接按目标仓位调整
    data_sorted.loc[i, 'actual_position'] = target
    current_position = data_sorted.loc[i, 'actual_position']

In [ ]:
data_sorted['position_change'] = data_sorted['actual_position'].diff().fillna(0)
data_sorted['daily_cost'] = 0.00

for i, row in data_sorted.iterrows():
    change = row['position_change']
    if change == 0:
        continue
    price = row['close']
    turnover = abs(change) * price
    
    if change > 0:      #买入：只付佣金和滑点
        cost = turnover * (COMMISSION_RATE + SLIPPAGE)
    else:               #卖出：付佣金、印花税和滑点
        cost = turnover * (COMMISSION_RATE + STAMP_TAX + SLIPPAGE)

    data_sorted.loc[i, 'daily_cost'] = cost

In [ ]:
data_sorted['strategy_return_raw'] = data_sorted['actual_position'] * data_sorted['return']
data_sorted['cost_return'] = data_sorted['daily_cost'] / (data_sorted['close'] * data_sorted['actual_position'].shift(1).replace(0, np.nan))
data_sorted['strategy_return'] = data_sorted['strategy_return_raw'] - data_sorted['cost_return'].fillna(0)
data_strategy = data_sorted.dropna().copy()
data_strategy['net_value'] = (1 + data_strategy['strategy_return']).cumprod()
data_strategy['benchmark'] = (1 + data_strategy['return']).cumprod()
#注意前期因为没有信号，策略净值可能会低于基准净值，这属于正常现象

In [ ]:
#绘制策略净值和基准净值的对比图
plt.figure(figsize=(12, 6))
plt.plot(data_strategy['net_value'], label='双均线策略收益', color='red', linewidth=2)
plt.plot(data_strategy['benchmark'], label='买入持有收益', color='blue', linestyle='--')
plt.title('双均线策略 vs 买入持有')
plt.xlabel('日期')
plt.ylabel('累计净值')
plt.legend()
plt.grid(True)
plt.show()